# Price sync

Fetches missing price history from Yahoo Finance for every symbol in the portfolio and updates `data/prices/`.
Each symbol gets its own cache file; only the missing date range is fetched on each run.

In [ ]:
from datetime import date

from trades.brokers.ibkr import main
from trades.config import AppConfig
from trades.ledger import replay
from trades.market_data import prices

AS_OF_DATE = date.today()
config = AppConfig()

In [ ]:
# Derive the symbol list and start date from the ledger
ledger = main.load_ledger(config)
result = replay.replay_ledger(ledger, config)

symbols = result.open_lots["symbol"].unique().sort().to_list()
first_trade_date = ledger["event_datetime"].min().date()
print(f"Symbols: {symbols}")
print(f"Fetching from {first_trade_date} to {AS_OF_DATE}")

In [ ]:
price_histories = prices.update_price_caches(symbols, since=first_trade_date, as_of=AS_OF_DATE, config=config)

for symbol, hist in price_histories.items():
    print(f"{symbol}: {len(hist)} trading days cached, latest close {hist['close'][-1]:.2f}")

In [ ]:
# Inspect one symbol's history
SYMBOL = symbols[0]
price_histories[SYMBOL]